<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 5: MCP Fundamentals

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Understand the tool integration problem** — why we need a universal standard for AI tools
2. **Learn MCP architecture** — Hosts, Clients, Servers, and transport mechanisms
3. **Explore the MCP ecosystem** — pre-built servers for common tasks
4. **Build your first MCP server** — create tools with FastMCP and `@mcp.tool()`
5. **Connect an agent to MCP** — use `MCPServerStdio` with OpenAI Agents SDK
6. **Discover tools automatically** — inspect tool names, descriptions, and schemas at runtime

## 1. Environment Setup

In [ ]:
!pip install -q openai-agents mcp pytz

In [ ]:
import os
import getpass
import json

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

In [ ]:
# Colab/Jupyter compatibility for MCP stdio transport
import sys, io, os

try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stderr = open(os.devnull, "w")

---

## 2. The Tool Integration Problem

Today, every AI agent framework has its own way of defining tools. The same tool must be written multiple times for different frameworks:

**Without MCP:**
```
Tool A  ──→  Framework 1 (custom format)
Tool A  ──→  Framework 2 (different format)
Tool A  ──→  Framework 3 (yet another format)
```

**With MCP:**
```
Tool A  ──→  MCP Server  ──→  Any Framework
```

MCP solves this by providing a **universal standard** — write your tool once as an MCP server, and any MCP-compatible agent can use it.

---

## 3. What is MCP?

**MCP (Model Context Protocol)** is an open standard that defines how AI agents connect to external tools, data sources, and services.

- **Open Standard**: Anyone can implement it — not locked to a single vendor
- **Universal**: Works across different agent frameworks (OpenAI Agents SDK, LangChain, etc.)
- **Created by Anthropic**: Developed by the makers of Claude
- **Growing Ecosystem**: Hundreds of pre-built MCP servers available
- **Automatic Discovery**: Agents learn what tools are available at runtime

---

## 4. MCP Architecture

MCP has three main components:

| Component | Role | Example |
|-----------|------|---------|  
| **Host** | The application containing the AI agent | Claude Desktop, VS Code, Your App |
| **Client** | Connects hosts to servers | Built into agent frameworks |
| **Server** | Provides tools, data, or prompts | File system, database, API wrappers |

### Server Capabilities

MCP servers can expose three types of capabilities:

- **Tools**: Actions the agent can perform (search, create, delete)
- **Resources**: Data the agent can read (files, database records)
- **Prompts**: Template instructions for specific tasks

### Transport Mechanisms

- **stdio**: Standard input/output streams — for local servers on the same machine
- **SSE**: Server-Sent Events over HTTP — for remote servers over the network

In this course, we'll focus on **stdio** transport.

> **Key insight:** The communication flow is: **Connect** → **Discover** (server returns available tools) → **Call Tool** (host requests execution) → **Return Result**. Agents don't need to know about tools in advance — they discover them at runtime.

---

## 5. The MCP Ecosystem

Pre-built MCP servers are available for many common tasks:

| Server | What it does |
|--------|-------------|  
| `@modelcontextprotocol/server-filesystem` | Read/write files on the local filesystem |
| `@anthropic/mcp-server-brave-search` | Search the web using Brave Search |
| `@playwright/mcp` | Browser automation and web scraping |
| `mcp-server-sqlite` | Query SQLite databases |
| `mcp-server-fetch` | Fetch and parse web pages |

Discover more at:
- [mcp.so](https://mcp.so) — Official MCP server directory
- [smithery.ai](https://smithery.ai/) — MCP server registry

---

## 6. Your First MCP Server

**FastMCP** makes it easy to create MCP servers using Python decorators. The pattern is:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("server_name")

@mcp.tool()
async def my_tool(param: str) -> str:
    """Tool description (used by agents to decide when to call it)."""
    return f"Result: {param}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

Let's build a datetime server:

In [ ]:
datetime_server_code = '''
from mcp.server.fastmcp import FastMCP
from datetime import datetime
import pytz

# Create the MCP server
mcp = FastMCP("datetime_server")

@mcp.tool()
async def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time in a specific timezone.
    
    Args:
        timezone: The timezone name (e.g., 'UTC', 'US/Eastern', 'Asia/Kolkata')
    """
    try:
        tz = pytz.timezone(timezone)
        current_time = datetime.now(tz)
        return f"Current time in {timezone}: {current_time.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except Exception as e:
        return f"Error: {str(e)}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("datetime_server.py", "w") as f:
    f.write(datetime_server_code)

print("Created datetime_server.py")
print("Tools: get_current_time")

> **Key insight:** The `@mcp.tool()` decorator turns any async function into an MCP tool. The docstring becomes the tool description that agents use to decide when to call it. Type hints define the parameter schema.

---

## 7. Connecting an Agent to MCP

The OpenAI Agents SDK provides `MCPServerStdio` for connecting to local MCP servers. The pattern is:

```python
async with MCPServerStdio(params={"command": "python", "args": ["server.py"]}) as server:
    agent = Agent(name="...", mcp_servers=[server])
```

In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerStdio
import asyncio

datetime_server_params = {
    "command": "python",
    "args": ["datetime_server.py"]
}

async def run_datetime_agent():
    async with MCPServerStdio(params=datetime_server_params, client_session_timeout_seconds=30) as datetime_server:
        print("Connected to datetime MCP server!")
        
        agent = Agent(
            name="TimeBot",
            instructions="""You are a helpful assistant that can tell time and 
            work with dates. Use the available tools to answer questions about 
            time and dates. Always be friendly and helpful.""",
            mcp_servers=[datetime_server],
            model="gpt-4o-mini"
        )
        
        result = await Runner.run(
            agent,
            "What time is it right now in India?"
        )
        
        print("\nAgent Response:")
        print(result.final_output)

await run_datetime_agent()

> **Key insight:** The agent automatically discovers the MCP tools and decides which ones to call based on the user's query. You don't need to manually wire up tool calls — MCP handles discovery, and the LLM handles selection.

---

## 8. Tool Discovery

One of MCP's most powerful features is **automatic tool discovery**. You can inspect exactly what tools a server provides — their names, descriptions, and parameter schemas:

In [ ]:
async def explore_mcp_tools():
    """Explore what tools are available from the MCP server."""
    
    async with MCPServerStdio(params=datetime_server_params, client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        
        print("TOOLS DISCOVERED FROM MCP SERVER")
        print("=" * 50)
        
        for tool in tools:
            print(f"\nTool: {tool.name}")
            print(f"Description: {tool.description}")
            print(f"Parameters: {json.dumps(tool.inputSchema, indent=2)}")

await explore_mcp_tools()

> **Key insight:** `server.list_tools()` returns structured metadata about every tool — this is what the agent uses to decide which tool to call. Good docstrings and type hints in your server code directly improve agent performance.

---

## 9. Multiple MCP Servers

A single agent can connect to **multiple MCP servers** simultaneously. Each server provides its own set of tools, and the agent can use tools from any of them.

Let's create a calculator server and connect it alongside our datetime server:

In [ ]:
calculator_server_code = '''
from mcp.server.fastmcp import FastMCP
import math

mcp = FastMCP("calculator_server")

@mcp.tool()
async def calculate(expression: str) -> str:
    """Evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression like "2 + 3 * 4" or "sqrt(16)"
    """
    try:
        allowed = {
            "sqrt": math.sqrt,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "log10": math.log10,
            "pi": math.pi,
            "e": math.e,
            "abs": abs,
            "pow": pow,
            "round": round
        }
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("calculator_server.py", "w") as f:
    f.write(calculator_server_code)

print("Created calculator_server.py")
print("Tools: calculate")

In [ ]:
calculator_server_params = {
    "command": "python",
    "args": ["calculator_server.py"]
}

async def run_multi_server_agent():
    """Run an agent connected to multiple MCP servers."""
    
    async with MCPServerStdio(params=datetime_server_params, client_session_timeout_seconds=30) as datetime_server:
        async with MCPServerStdio(params=calculator_server_params, client_session_timeout_seconds=30) as calc_server:
            
            # List tools from both servers
            datetime_tools = await datetime_server.list_tools()
            calc_tools = await calc_server.list_tools()
            
            print("AVAILABLE TOOLS FROM MULTIPLE SERVERS:")
            print("\nFrom datetime_server:")
            for tool in datetime_tools:
                print(f"  - {tool.name}")
            print("\nFrom calculator_server:")
            for tool in calc_tools:
                print(f"  - {tool.name}")
            
            # Create agent with BOTH servers
            agent = Agent(
                name="MultiToolBot",
                instructions="""You are a helpful assistant with access to 
                datetime tools and calculation tools. Use the appropriate 
                tools to answer user questions.""",
                mcp_servers=[datetime_server, calc_server],
                model="gpt-4o-mini"
            )
            
            query = """What time is it right now in New York? 
            Also, what is the square root of 144?"""
            
            print("\n" + "=" * 50)
            print(f"Query: {query}")
            print("=" * 50)
            
            result = await Runner.run(agent, query)
            print(f"\nResponse:\n{result.final_output}")

await run_multi_server_agent()

> **Key insight:** When an agent has multiple MCP servers, it sees ALL tools from ALL servers. The LLM decides which tools to call based on the query — it might use tools from different servers in a single response.

---

## 10. Key Takeaways

| Concept | What it does |
|---------|-------------|
| **MCP** | Open standard for connecting AI agents to tools — write once, use everywhere |
| **Host / Client / Server** | Architecture: your app → MCP client → MCP server |
| **Tools / Resources / Prompts** | Three capabilities a server can expose |
| **FastMCP** | Python library for creating servers with `@mcp.tool()` decorators |
| **MCPServerStdio** | OpenAI Agents SDK class for connecting to local MCP servers |
| **Tool Discovery** | `server.list_tools()` — agents learn available tools at runtime |
| **Multi-server** | One agent can connect to multiple MCP servers simultaneously |

---

## 11. Exercises

### Exercise 1: Weather MCP Server
Create an MCP server with tools for weather information (use mock data). Include tools for `get_weather(city)`, `get_forecast(city, days)`, and `compare_weather(city1, city2)`. Connect it to an agent and test with several queries.

### Exercise 2: Multi-Server Assistant
Connect your weather server alongside the datetime and calculator servers to create a travel planning assistant. Test queries like: *"I'm traveling from Delhi to New York tomorrow. What's the time difference, weather in both cities, and what's 50,000 INR in USD?"*

---

**What's Next?** In the next notebook, we'll build MCP clients, create specialized servers, and assemble a complete research assistant with a Gradio interface.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---